# Analyse du Marché de l'Emploi IT au Maroc - Mexora RH

Ce notebook présente les résultats de l'analyse du marché de l'emploi IT marocain pour aider Mexora à structurer sa stratégie de recrutement. L'analyse est réalisée avec DuckDB en requêtant directement les données de la zone Gold de notre Data Lake.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration graphique
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Chemins vers les données
GOLD = '../data_lake/gold'
SILVER = '../data_lake/silver'

# Connexion DuckDB
con = duckdb.connect()

## Question 1 — Quelles compétences sont les plus demandées au Maroc en IT ?

In [ ]:
df_top20 = con.execute(f"""
    SELECT famille, competence, 
           SUM(nb_offres_mentionnent) AS nb_offres, 
           ROUND(AVG(pct_offres_total), 2) AS pct
    FROM '{GOLD}/top_competences.parquet'
    WHERE competence != 'non_detecte'
    GROUP BY famille, competence
    ORDER BY nb_offres DESC
    LIMIT 20
""").df()

plt.figure(figsize=(10, 8))
sns.barplot(data=df_top20, y='competence', x='nb_offres', hue='famille', dodge=False)
plt.title('Top 20 des compétences IT les plus demandées', fontweight='bold')
plt.xlabel('Nombre d\'offres')
plt.ylabel('')
plt.legend(title='Famille', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Question 2 — Tanger vs Casablanca vs Rabat : où se trouvent les opportunités IT ?

In [ ]:
df_villes = con.execute(f"""
    SELECT ville, 
           SUM(nb_offres) AS total_offres, 
           ROUND(SUM(nb_offres_remote)*100.0/NULLIF(SUM(nb_offres),0),1) AS pct_remote
    FROM '{GOLD}/offres_par_ville.parquet'
    GROUP BY ville
    ORDER BY total_offres DESC
    LIMIT 8
""").df()

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.bar(df_villes['ville'], df_villes['total_offres'], color='skyblue', label='Volume total')
ax1.set_ylabel('Nombre d\'offres', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

ax2 = ax1.twinx()
ax2.plot(df_villes['ville'], df_villes['pct_remote'], color='red', marker='o', linewidth=2, label='% Remote/Hybride')
ax2.set_ylabel('% Télétravail / Hybride', color='red')
ax2.tick_params(axis='y', labelcolor='red')
ax2.set_ylim(0, 100)

plt.title('Volume d\'offres et flexibilité par ville', fontweight='bold')
plt.tight_layout()
plt.show()

## Question 3 — Quel est le salaire médian par profil IT au Maroc ?

In [ ]:
df_sal = con.execute(f"""
    SELECT profil, 
           ROUND(MEDIAN(salaire_median_mad),0) AS salaire_median,
           ROUND(MIN(salaire_min_observe),0) AS plancher,
           ROUND(MAX(salaire_max_observe),0) AS plafond
    FROM '{GOLD}/salaires_par_profil.parquet'
    GROUP BY profil
    ORDER BY salaire_median DESC
""").df()

plt.figure(figsize=(12, 6))
sns.barplot(data=df_sal, y='profil', x='salaire_median', color='teal')
plt.title('Salaire Médian Mensuel par Profil (MAD)', fontweight='bold')
plt.xlabel('Salaire Médian (MAD)')
plt.ylabel('')
plt.tight_layout()
plt.show()

display(df_sal)

## Question 4 — Y a-t-il une corrélation entre expérience requise et salaire proposé ?

In [ ]:
df_corr = con.execute(f"""
    SELECT profil_normalise AS profil, 
           ROUND(CORR(CAST(experience_min_ans AS DOUBLE), CAST(salaire_median_mad AS DOUBLE)),3) AS correlation_pearson
    FROM '{SILVER}/offres_clean/offres_clean.parquet'
    WHERE salaire_connu=true AND experience_min_ans IS NOT NULL AND salaire_median_mad IS NOT NULL
    GROUP BY profil_normalise
    ORDER BY correlation_pearson DESC
""").df()

display(df_corr)

## Question 5 — Quelles entreprises recrutent le plus ? Qui sont les concurrents ?

In [ ]:
df_ent = con.execute(f"""
    SELECT entreprise, ville, nb_offres_publiees, salaire_moyen_propose
    FROM '{GOLD}/entreprises_recruteurs.parquet'
    ORDER BY nb_offres_publiees DESC
    LIMIT 15
""").df()

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if v == 'Tanger' else '#3498db' for v in df_ent['ville']]
plt.barh(df_ent['entreprise'][::-1], df_ent['nb_offres_publiees'][::-1], color=colors[::-1])
plt.title('Top 15 Entreprises Recruteurs (Rouge = Tanger)', fontweight='bold')
plt.xlabel('Nombre d\'offres')
plt.tight_layout()
plt.show()

## Dashboard de Synthèse Final
Regroupement des 4 visualisations stratégiques (Carte, Boxplot, Top 15 Compétences, Tendances mensuelles) en une seule figure.

In [ ]:
import matplotlib.patches as mpatches
import numpy as np

COLORS = {
    'langages': '#4361EE', 'frameworks_web': '#3A0CA3', 'data_engineering': '#F72585',
    'cloud': '#7209B7', 'bi_analytics': '#560BAD', 'devops_infra': '#480CA8',
    'databases': '#3F37C9', 'data_science_ml': '#4CC9F0', 'methodologies': '#4895EF',
    'securite': '#B5179E', 'autres': '#F3722C',
}
PROFIL_COLORS = {
    'Data Engineer': '#F72585', 'Data Analyst': '#4361EE', 'Data Scientist': '#7209B7',
    'Developpeur Full Stack': '#4CC9F0', 'Developpeur Backend': '#3A0CA3',
    'Developpeur Frontend': '#4895EF', 'DevOps / SRE': '#F3722C',
    'Cloud Engineer': '#560BAD', 'Developpeur Mobile': '#3F37C9',
    'Cybersecurite': '#B5179E', 'Chef de Projet IT': '#023E8A', 'Autre IT': '#888',
}

# Extractions des donnees pour le Dashboard
df_comp = con.execute(f"""
    SELECT famille, competence, SUM(nb_offres_mentionnent) AS nb
    FROM '{GOLD}/top_competences.parquet' WHERE competence!='non_detecte'
    GROUP BY famille, competence ORDER BY nb DESC LIMIT 15
""").df()

df_tendances = con.execute(f"""
    SELECT annee||'-'||mois AS periode, profil, SUM(nb_offres) AS nb
    FROM '{GOLD}/tendances_mensuelles.parquet'
    WHERE profil IN ('Data Engineer','Data Analyst','Data Scientist')
      AND annee IS NOT NULL
    GROUP BY annee, mois, profil ORDER BY annee, mois
""").df()

df_villes_map = con.execute(f"""
    SELECT ville, SUM(nb_offres) AS nb FROM '{GOLD}/offres_par_ville.parquet'
    GROUP BY ville ORDER BY nb DESC LIMIT 10
""").df()

df_box2 = con.execute(f"""
    SELECT profil_normalise AS profil, salaire_median_mad
    FROM '{SILVER}/offres_clean/offres_clean.parquet'
    WHERE salaire_connu=true AND salaire_median_mad IS NOT NULL
      AND profil_normalise IN ('Data Engineer','Data Analyst','Data Scientist',
          'Developpeur Full Stack','DevOps / SRE')
""").df()

fig = plt.figure(figsize=(20, 14))
fig.suptitle("Dashboard - Marché de l'Emploi IT au Maroc | Mexora RH Intelligence 2024",
             fontsize=18, fontweight='bold', y=0.98)

# 1. Carte du Maroc (bubble chart villes)
ax1 = fig.add_subplot(2, 2, 1)
ville_coords = {
    'Casablanca': (0.35, 0.35), 'Rabat': (0.32, 0.55), 'Tanger': (0.25, 0.78),
    'Marrakech': (0.30, 0.20), 'Fes': (0.52, 0.60), 'Agadir': (0.18, 0.12),
    'Oujda': (0.78, 0.55), 'Meknes': (0.44, 0.58), 'Kenitra': (0.30, 0.60),
    'Tetouan': (0.28, 0.82),
}
for _, row in df_villes_map.iterrows():
    v = row['ville']
    if v in ville_coords:
        x, y = ville_coords[v]
        size = max(100, row['nb'] * 0.4)
        c = '#F72585' if v == 'Tanger' else '#4361EE'
        ax1.scatter(x, y, s=size, color=c, alpha=0.7)
        ax1.annotate(f"{v}\n({row['nb']})", (x, y), textcoords="offset points",
                    xytext=(5, 5), fontsize=10, fontweight='bold')
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
ax1.set_title("Volume d'offres IT par ville", fontweight='bold', fontsize=14)
ax1.axis('off')
ax1.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, edgecolor='#ccc', lw=2))

# 2. Top 15 competences
ax2 = fig.add_subplot(2, 2, 2)
bar_colors2 = [COLORS.get(f, '#888') for f in df_comp['famille']]
ax2.barh(df_comp['competence'][::-1], df_comp['nb'][::-1], color=bar_colors2[::-1])
ax2.set_title("Top 15 compétences IT (par famille)", fontweight='bold', fontsize=14)
ax2.set_xlabel("Nombre d'offres")

# 3. Boxplot salaires
ax3 = fig.add_subplot(2, 2, 3)
profils_ord2 = df_box2.groupby('profil')['salaire_median_mad'].median().sort_values(ascending=False).index.tolist()
data_bp = [df_box2[df_box2['profil']==p]['salaire_median_mad'].dropna().values for p in profils_ord2]
bp2 = ax3.boxplot(data_bp, labels=[p.replace('Developpeur ','Dev. ') for p in profils_ord2],
                  patch_artist=True)
for patch, p in zip(bp2['boxes'], profils_ord2):
    patch.set_facecolor(PROFIL_COLORS.get(p, '#888'))
    patch.set_alpha(0.7)
ax3.set_title("Distribution des salaires par profil (MAD/mois)", fontweight='bold', fontsize=14)
ax3.set_ylabel("MAD/mois")
ax3.tick_params(axis='x', rotation=15)
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{int(v):,}"))

# 4. Tendances mensuelles
ax4 = fig.add_subplot(2, 2, 4)
for profil, grp in df_tendances.groupby('profil'):
    ax4.plot(range(len(grp)), grp['nb'], marker='o', markersize=5,
             label=profil, color=PROFIL_COLORS.get(profil,'#888'), linewidth=2.5)
periodes = df_tendances['periode'].unique()
step = max(1, len(periodes)//8)
ax4.set_xticks(range(0, len(periodes), step))
ax4.set_xticklabels(list(periodes)[::step], rotation=30, fontsize=10)
ax4.set_title("Évolution mensuelle des offres Data (2023-2024)", fontweight='bold', fontsize=14)
ax4.set_ylabel("Nombre d'offres")
ax4.legend(fontsize=11)
ax4.grid(alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()